In [4]:
import pandas as pd
import geopandas as gpd
import duckdb

In [5]:
#connect to duckdb
con = duckdb.connect('../database/spatial-db.db', read_only=False)

In [6]:
#install the spatial extension
con.install_extension("spatial")
con.load_extension("spatial")

In [7]:
states = gpd.read_file('data/00_state/state.shp')

In [8]:
states.head()

,GEOID,state_name,ppl_densit,transit_to,walk_to_wo,c_lat,c_lon,neighbors_,state_tot_,tot_cov_po,...,occ_housin,gas,electricit,oil,pct_gas,pct_electr,pct_oil,main_fuel,a_neighbor,geometry
0,1,Alabama,99.26968,0.003771,0.011017,32.756880,-86.844521,"['Tennessee', 'Florida', 'Mississippi', 'Georg...",5074296,4413000,...,1933150,611832,1290659,2319,31.6,66.8,0.1,electricity,"['Florida', 'Georgia', 'Mississippi', 'Tenness...","POLYGON ((-85.12733 31.76256, -85.12753 31.762..."
1,4,Arizona,63.10557,0.014370,0.016875,34.293141,-111.664444,"['Utah', 'none', 'California', 'New Mexico']",7359197,5919000,...,2739136,987027,1644807,2607,36.0,60.0,0.1,electricity,"['California', 'Colorado', 'Nevada', 'New Mexi...","POLYGON ((-110.75069 37.00301, -110.74193 37.0..."
2,5,Arkansas,58.05936,0.003333,0.015007,34.899917,-92.438374,"['Missouri', 'Louisiana', 'Oklahoma', 'Mississ...",3045637,2677000,...,1171694,513031,614807,1123,43.8,52.5,0.1,electricity,"['Louisiana', 'Mississippi', 'Missouri', 'Okla...","POLYGON ((-90.95577 34.11871, -90.95451 34.117..."
3,6,California,252.51050,0.038160,0.023834,37.215333,-119.663871,"['Oregon', 'none', 'none', 'Nevada']",39029342,33840000,...,13315822,8750976,3769978,30558,65.7,28.3,0.2,gas,"['Arizona', 'Nevada', 'Oregon']","MULTIPOLYGON (((-119.99987 41.18397, -119.9998..."
4,8,Colorado,55.68269,0.022252,0.026464,38.998532,-105.547821,"['Wyoming', 'New Mexico', 'Utah', 'Nebraska']",5839926,4291000,...,2278044,1638883,567786,2640,71.9,24.9,0.1,gas,"['Arizona', 'Kansas', 'Nebraska', 'New Mexico'...","POLYGON ((-105.15504 36.99526, -105.15543 36.9..."


In [9]:
states.loc[states['state_name'] == 'Tennessee', 'neighbors_']

40    [Kentucky, Alabama, Arkansas, North Carolina]
Name: neighbors_, dtype: object

In [12]:
states.loc[states['state_name'] == 'Mississippi', 'neighbors_']

22    [Tennessee, Louisiana, Arkansas, Alabama]
Name: neighbors_, dtype: object

In [11]:
states.at[22, 'neighbors_'] = ['Tennessee', 'Louisiana', 'Arkansas', 'Alabama']

In [8]:
states.at[40, 'neighbors_'] = ['Kentucky', 'Alabama', 'Arkansas', 'North Carolina']

In [13]:
#save to shapefile
states.to_file('data/00_state/state.shp')

In [9]:
#read the shapefile into duckdb
con.sql("SELECT * FROM ST_Read('data/00_state/state.shp')")

┌───────┬──────────────────────┬────────────┬───────────────────┬───────────────────┬────────────────────┬─────────────────────┬──────────────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬────────────┬────────┬────────────────────┬────────────────────┬────────────────────┬─────────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [10]:
#drop table if it exists
con.execute("DROP TABLE IF EXISTS state")

In [11]:
#create a table for the state shapefile
con.sql("CREATE TABLE state AS SELECT * FROM ST_Read('data/00_state/state.shp')")

In [12]:
#check if the table was created
con.table('state')

┌───────┬──────────────────────┬────────────┬───────────────────┬───────────────────┬────────────────────┬─────────────────────┬──────────────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬────────────┬────────┬────────────────────┬────────────────────┬────────────────────┬─────────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [13]:
#get all the columns in the table
con.table('state').columns

['GEOID',
 'state_name',
 'ppl_densit',
 'transit_to',
 'walk_to_wo',
 'c_lat',
 'c_lon',
 'neighbors_',
 'state_tot_',
 'tot_cov_po',
 'pct_tot_co',
 'pct_no_fix',
 'no_bb_or_c',
 'pct_no_bb_',
 'occ_housin',
 'gas',
 'electricit',
 'oil',
 'pct_gas',
 'pct_electr',
 'pct_oil',
 'main_fuel',
 'a_neighbor',
 'geom']

In [14]:
con.close()